# Xenium Prime 5K Breast — expression-only GNN communication niches

Select the **cccphe** kernel and use **Run All**. This notebook invokes existing project modules to prepare and audit the real inputs, train the current model when needed, and display its results. A complete result with matching configuration is reused. New outputs are independent of archived AE/GNN results and remain on NAS. This is an independent downstream single-tissue analysis; no clinical/phenotype/TLS supervision enters the clinical CCC/Cox mainline.

Current training uses CCC reconstruction + program regularization only (lambda_comp=0, lambda_reg=0.005). Cell-type composition reconstruction is disabled. Restart Kernel / Run All to use this configuration. Results are stored in an independent NAS directory; archived composition-constrained models remain unchanged.

In [1]:
%matplotlib inline
from pathlib import Path
import sys,json
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path('/home/xueshuailin/CCC_Phe')
sys.path.insert(0,str(ROOT/'src'))
from phenoniche.gnn.config import GNNConfig
from phenoniche.ccc.config import default_config
from phenoniche.gnn.pipeline import ensure_results
from phenoniche.gnn.reporting import GNNReport
CONFIG=ROOT/'config/gnn.json'
config=GNNConfig.from_json(CONFIG)
OUTPUT=Path(config.output_dir)
pd.set_option('display.max_colwidth',150)
pd.set_option('display.max_rows',200)
def show(fig):
    display(fig)
    plt.close(fig)

## 1. Real data and model configuration

The current LR expression coverage filter is 5%. The encoder receives expression and geometry only. Cell types supply directed CCC labels and downstream annotations only; they are not encoder inputs or composition reconstruction supervision. Expression normalization, feature scaling, and the effective CCC mask are fitted only on training spatial blocks. No CollecTRI/NicheNet supervision or regulatory target holdout is used.

In [2]:
input_config=default_config()
adata=ad.read_h5ad(input_config.processed_h5ad,backed='r')
try:
    display(pd.Series({'cells':adata.n_obs,'panel_genes':adata.n_vars,'dataset':adata.uns.get('dataset'),'expression_coverage_fraction':input_config.expression_coverage_fraction}).to_frame('Input'))
finally:
    adata.file.close()
display(pd.Series(config.to_dict()).to_frame('Model configuration'))
print('NAS outputs:',OUTPUT)

,Input
cells,699110
panel_genes,5101
dataset,Xenium_Prime_Breast_Cancer_FFPE
expression_coverage_fraction,0.05


,Model configuration
seed,40700
split_seed,40700
programs,32
hidden,64
encoder_genes,512
batch_size,512
max_epochs,40
patience,6
lr,0.003
weight_decay,0.0001


NAS outputs: /home/nas3/biod/xueshuailin/CCC_Phe/workspace/outputs/microenvironment_gnn_ccc_only_expr05


## 2. Prepare, audit, train/reuse, and infer

Run All constructs/audits the unchanged directed CCC target and applies the repository audit gate. Spatial splits purge shared physical cells. Matching complete results and compatible checkpoints are reused; archived incompatible checkpoints are protected.

Z = aP, P = Softmax(f(G)), C_hat = ZH. H is nonnegative with unit row sums.

L = L_CCC + 0.005 L_regularization. There is no cell-type composition reconstruction loss. Cell composition is used for descriptive downstream diagnostics only.

In [ ]:
summary=ensure_results(CONFIG)
report=GNNReport(OUTPUT)
display(report.overview())
display(report.features.head(12))
display(pd.Series(summary).to_frame('Saved analysis summary'))

## 3. Training curves and held-out CCC reconstruction

Train/validation/test centers do not share physical cells. Only CCC reconstruction and regularization contribute to the training objective.

In [ ]:
show(report.training_curves())
show(report.reconstruction())
display(pd.DataFrame(report.reconstruction_metrics).T)
display(pd.Series(json.loads((OUTPUT/'heldout_metrics.json').read_text())).to_frame('Held-out evaluation'))

## 4. All 32 communication programs and CCC dictionary H

Absolute Z preserves total communication activity; P describes relative communication-program activity. H traces each program to sender → receiver → LR features. The H heatmap highlights representative features; full H is exported as CSV. No composition decoder is trained or shown.

In [ ]:
fig,usage=report.program_usage();show(fig);display(usage)
show(report.dictionary_heatmap())
display(pd.read_csv(OUTPUT/'programs/top_ccc.csv').groupby('program').head(5))

## 5. Select K = 4–15 and identify communication-defined niches

Clustering is based on communication-program mixture and activity from Z. K is fitted on training blocks and selected on validation blocks; test blocks are reserved for final diagnostics. Seed stability refers to clustering seeds, not independent GNN retraining.

In [ ]:
show(report.k_selection())
display(report.selection)

## 6. Tissue-space maps, individual niches, composition and enrichment

All cell centers use original micrometer coordinates. Enrichment is descriptive within one tissue; overlapping microenvironments are not independent biological replicates. The local composition plot uses the exact pair-weighted composition target.

In [ ]:
show(report.spatial_overview())
show(report.each_niche())
fig,fraction,enrichment=report.composition();show(fig)
display(fraction)
display(enrichment)
show(report.local_composition())

## 7. Niche × program activity and representative communication

Activity heatmaps use mean Z of assigned cells. Representative niche CCC profiles use training-block soft program contributions, preserving the trained H interpretation. Cell composition is downstream descriptive analysis, not an auxiliary reconstruction task or evidence of causality.

In [ ]:
fig,activity=report.niche_programs();show(fig);display(activity)
overview,ccc=report.interpretation();display(overview)
for niche in range(1,report.k+1):
    print(f'Niche {niche}: representative sender–receiver–LR')
    display(ccc.loc[ccc.niche==niche].head(5))
display(pd.Series(json.loads((OUTPUT/'diagnostics.json').read_text())).to_frame('Diagnostic'))

## 8. Comparison with archived CCC-vector AE

Archived AE weights/results remain unchanged. Spatial maps and cell assignment correspondence are descriptive: the archived model used overlapping random folds and a narrower CCC feature universe. This comparison does not establish comparative generalization or phenotype associations.

In [ ]:
fig,comparison=report.comparison();show(fig);display(comparison)
print('Model arrays, native CCC reconstruction, records, tables and PNG/PDF:',OUTPUT)